# Live Data Fetchers for KMDW Trading

Real-time data ingestion layer for the live trading pipeline. Each fetcher
returns a DataFrame with the **same schema as the corresponding historical
parquet** (`data/{metar,taf,asos,cli}_KMDW.parquet`), so the §3 feature
engineering cells from `model_v3.ipynb` / `model_v4.ipynb` work unchanged
on the live data.

| Source | Endpoint | Latency | Returns |
|---|---|---|---|
| **METAR** | `api.weather.gov/stations/KMDW/observations` | **sub-minute** | Last N METAR/SPECI observations, 0.1°C T-group |
| **TAF** | IEM `taf.py` | ~10 min | Pre-parsed TAF segments |
| **ASOS 1-min** | IEM `asos1min.py` | **~30-60 min** | 1-min cadence, whole °F (latency caveat) |
| **HRRR** | Herbie/AWS S3 | ~50-90 min after init | Gridded NWP forecast for KMDW grid cell |
| **CLI** | IEM `cli.py` | Next morning after settlement | Daily climate report (settlement / persistence) |
| **Kalshi markets** | `api.elections.kalshi.com` | Real-time | Today's 6 KXHIGHCHI contracts: strikes, bids/asks |

**Reuses `refresh_data.py` fetchers** where they're already real-time-capable
(TAF, ASOS, CLI). Adds new fetchers for METAR (via api.weather.gov for lower
latency), HRRR (via Herbie), and Kalshi (new endpoint).

**Authentication:** none of these require auth for read-only access. Kalshi
order placement (Phase 2) requires a paid account + API key — not in this
notebook.

## §1 — Setup

In [1]:
import sys
import time
import requests
import pandas as pd
import numpy as np
from io import BytesIO
from datetime import datetime, timezone, timedelta
from pathlib import Path
from metar import Metar

# Import the historical fetchers from refresh_data.py so we re-use the exact
# same parsing logic (guarantees schema parity with the trained models).
sys.path.insert(0, str(Path.cwd()))
from refresh_data import (
    fetch_metar     as _hist_fetch_metar,
    fetch_taf       as _hist_fetch_taf,
    fetch_asos_1min as _hist_fetch_asos_1min,
    fetch_cli       as _hist_fetch_cli,
    _parse_metar,
)

STATION    = "KMDW"
DATA_DIR   = Path("data")
USER_AGENT = "weather-alpha-research/0.1 (kunpark04@gmail.com)"

KALSHI_API     = "https://api.elections.kalshi.com/trade-api/v2"
WEATHER_GOV    = "https://api.weather.gov"

NOW_UTC = datetime.now(timezone.utc)
print(f"Live fetcher setup for {STATION} at {NOW_UTC.isoformat()}")


def _coerce_to_reference(df, reference_parquet):
    """Cast df columns in-place to match the reference parquet's dtypes.

    Idempotent; per-column try/except so a single bad cast doesn't poison
    the rest. Used by each fetch_live_* function to guarantee dtype parity
    with the historical parquet even when the live sample is small (where
    pandas would otherwise infer object dtype for all-None columns).

    Args:
    - df (pd.DataFrame): live DataFrame to coerce.
    - reference_parquet (str | Path): path to historical parquet whose
      dtypes define the target schema. If missing, df is returned unchanged.

    Returns:
    - pd.DataFrame: same object, columns cast to target dtypes where possible.
    """
    try:
        ref = pd.read_parquet(reference_parquet)
    except FileNotFoundError:
        return df
    for col in df.columns:
        if col not in ref.columns:
            continue
        target = ref[col].dtype
        if df[col].dtype == target:
            continue
        try:
            df[col] = df[col].astype(target)
        except (ValueError, TypeError):
            pass
    return df


Live fetcher setup for KMDW at 2026-05-22T06:40:38.409975+00:00


## §2.1 — Live METAR (api.weather.gov)

Pulls the most recent METAR/SPECI observations directly from NWS via
`api.weather.gov`. Sub-minute latency, includes the 0.1°C T-group temperature
in `rawMessage`.

**Schema parity trick:** instead of parsing api.weather.gov's structured
fields (which would diverge from how IEM-fetched historical METARs were
parsed), we extract the `rawMessage` field and feed it through the **same
`_parse_metar()` function** used for the historical parquet. Identical output
columns, no train-serve skew.

In [2]:
def _parse_api_props(props):
    """Extract MET fields from api.weather.gov's parsed properties dict.

    Mirrors _parse_metar() so the returned row matches metar_KMDW.parquet
    schema. Used for 5-minute observations that lack rawMessage. Remarks-only
    fields (max6/min6/max24/min24/peak_wind/precip/snow) stay None.

    Unit assumptions (api.weather.gov geo+json defaults):
      windSpeed/windGust in km/h -> knots, visibility in m -> SM,
      barometricPressure/seaLevelPressure in Pa -> mb.
    """
    def _val(key):
        d = props.get(key)
        return d.get("value") if isinstance(d, dict) else None

    def _kmh_to_kt(v):
        return None if v is None else round(v * 0.539957)

    def _m_to_sm(v):
        return None if v is None else v / 1609.344

    def _pa_to_mb(v):
        return None if v is None else v / 100.0

    return {
        "report_type":  None,
        "mod":          None,
        "wdir":         _val("windDirection"),
        "wspd_kt":      _kmh_to_kt(_val("windSpeed")),
        "gust_kt":      _kmh_to_kt(_val("windGust")),
        "wdir_from":    None,
        "wdir_to":      None,
        "peak_wspd_kt": None,
        "peak_wdir":    None,
        "vis_sm":       _m_to_sm(_val("visibility")),
        "wx":           None,
        "sky":          None,
        "temp_c":       _val("temperature"),
        "dewp_c":       _val("dewpoint"),
        "max6_c":       None,
        "min6_c":       None,
        "max24_c":      None,
        "min24_c":      None,
        "press_mb":     _pa_to_mb(_val("barometricPressure")),
        "slp_mb":       _pa_to_mb(_val("seaLevelPressure")),
        "p1hr_in":      None,
        "p3hr_in":      None,
        "p6hr_in":      None,
        "p24hr_in":     None,
        "snow_in":      None,
    }


def fetch_live_metar(station=STATION, limit=40):
    """Fetch the last `limit` observations from api.weather.gov, returning both
    a T-group subset (rawMessage rows, 0.1C precision METARs) and the full
    5-minute obs stream.

    api.weather.gov interleaves standard hourly METAR/SPECI reports (which
    have rawMessage and can be parsed by python-metar at 0.1C precision) with
    5-minute automatic obs that lack rawMessage but expose temp/dewpoint/wind/
    pressure/visibility through their API properties dict. The previous
    implementation dropped all rows without rawMessage, losing ~23 of every
    24 observations. This version returns both:

        - "tgroup":  only rows that had rawMessage. Parsed via _parse_metar()
          so 0.1C precision and remarks-derived fields (6h/24h temp extremes,
          peak wind, precip totals) are populated when present.
        - "obs5min": ALL rows. 5-min rows have temp_c, dewp_c, wind, pressure,
          visibility from API properties; remarks-only fields stay NaN.

    Both DataFrames share the metar_KMDW.parquet schema and are sorted by
    timestamp ascending.

    Args:
    - station (str): full ICAO (e.g., 'KMDW').
    - limit (int): number of most-recent observations to fetch. Default 40.

    Returns:
    - dict {"tgroup": pd.DataFrame, "obs5min": pd.DataFrame}.
      Each may be empty if api.weather.gov returns nothing.
    """
    url = f"{WEATHER_GOV}/stations/{station}/observations?limit={limit}"
    resp = requests.get(
        url,
        headers={"User-Agent": USER_AGENT, "Accept": "application/geo+json"},
        timeout=30,
    )
    resp.raise_for_status()
    payload = resp.json()

    rows = []
    for feat in payload.get("features", []):
        props = feat.get("properties", {})
        ts = pd.to_datetime(props["timestamp"], utc=True)
        raw = props.get("rawMessage")
        if raw:
            parsed = _parse_metar(raw)
            parsed["_has_raw"] = True
        else:
            parsed = _parse_api_props(props)
            parsed["_has_raw"] = False
        rows.append({"station": station, "timestamp": ts, **parsed})

    df = pd.DataFrame(rows)
    if df.empty:
        return {"tgroup": df, "obs5min": df}

    # Match Int64 casts in refresh_data.fetch_metar so dtype parity holds.
    for c in ["wdir", "wspd_kt", "gust_kt", "wdir_from", "wdir_to",
              "peak_wspd_kt", "peak_wdir", "snow_in"]:
        if c in df.columns:
            df[c] = df[c].astype("Int64")
    df = df.sort_values("timestamp").reset_index(drop=True)

    ref = DATA_DIR / f"metar_{station}.parquet"
    tgroup  = _coerce_to_reference(
        df[df["_has_raw"]].drop(columns=["_has_raw"]).reset_index(drop=True),
        ref,
    )
    obs5min = _coerce_to_reference(df.drop(columns=["_has_raw"]), ref)
    return {"tgroup": tgroup, "obs5min": obs5min}


## §2.2 — Live TAF (IEM)

api.weather.gov TAF endpoints return raw text products that would need custom
parsing to match our historical TAF schema (one row per FROM/BECMG/TEMPO
segment). IEM already serves this pre-parsed via CSV, with ~10-minute latency.
Simpler to use IEM and accept the modest lag — TAFs only cycle 4× per day so
10 minutes is negligible relative to the 6-hour cycle.

In [3]:
def fetch_live_taf(station=STATION, hours_back=36):
    """Fetch recent TAFs from IEM (last `hours_back` hours).

    Args:
    - station (str): full ICAO (e.g., 'KMDW'). The 'K' prefix is stripped
      internally because IEM's TAF endpoint uses the 3-letter FAA suffix.
    - hours_back (int): how far back to fetch. Default 36h ensures we
      always capture at least one full TAF cycle (6h) plus a buffer.

    Returns:
    - pd.DataFrame with the same columns as data/taf_KMDW.parquet.
    """
    now = datetime.now(timezone.utc)
    start = now - timedelta(hours=hours_back)
    bare = station[1:] if station.startswith("K") else station
    df = _hist_fetch_taf(
        start.strftime("%Y-%m-%dT%H:%MZ"),
        now.strftime("%Y-%m-%dT%H:%MZ"),
        bare,
    )
    return _coerce_to_reference(df, DATA_DIR / f"taf_{station}.parquet")


## §2.3 — Live ASOS 1-min (IEM)

**Latency caveat:** IEM's ASOS 1-min feed runs ~30–60 min behind real time
(sometimes 1–2 hours). For on-the-hour anchor predictions (e.g., predict at
6:00 AM CDT), the last ~1 hour of ASOS data may be missing — features like
`asos_temp_f_T` and `running_max_F_T` may not reflect the most recent observed
state.

**Mitigation:** if the model's ASOS-derived features matter for live decisions
(they do for the running max), consider either:
  (a) Accepting the lag (simplest, no skew vs training)
  (b) Reconstructing the missing recent hour from METAR T-group (sub-hourly
      cadence during SPECIs, but only hourly otherwise)
  (c) Skipping the ASOS-derived features entirely in the live path

For now: just return what IEM has and let the downstream feature engineering
handle missing-data rows.

In [4]:
def fetch_live_asos_1min(station=STATION, hours_back=48):
    """Fetch recent ASOS 1-min observations from IEM.

    Args:
    - station (str): full ICAO (e.g., 'KMDW').
    - hours_back (int): how far back to fetch. Default 48h is enough to
      always cover yesterday's full day + today's morning hours
      (accounting for IEM's typical 30-60 min lag).

    Returns:
    - pd.DataFrame matching data/asos_KMDW.parquet schema.
    """
    now = datetime.now(timezone.utc)
    start = now - timedelta(hours=hours_back)
    df = _hist_fetch_asos_1min(
        start.strftime("%m/%d/%Y"),
        (now + timedelta(days=1)).strftime("%m/%d/%Y"),   # exclusive upper bound
        station,
    )
    return _coerce_to_reference(df, DATA_DIR / f"asos_{station}.parquet")


## §2.4 — Live HRRR (Herbie/AWS S3)

Three functions in this section, low-level → high-level:

1. **`fetch_live_hrrr(init_dt, fxx_list, variables)`** — raw fetch for a specific run + lead-hour list. Caller picks the init.
2. **`fxx_for_peak_window(init_dt, ...)`** — derives the lead-hours covering noon–4 PM local of the init's local date. DST-aware.
3. **`latest_safe_hrrr_init(now_utc, publish_lag_min=90, synoptic_only=True)`** — picks the latest synoptic run guaranteed published given a 90-min publish lag.
4. **`fetch_hrrr_for_anchor(anchor_dt_local, ...)`** — top-level wrapper: combines all three to "give me the HRRR peak-window forecast for the prediction made at this anchor time in local clock, using the safest available run."

The anchor-aware wrapper is what you'd call from the live-prediction notebook. Caller doesn't need to know about UTC, synoptic vs hourly inits, publish lag, or peak-window arithmetic.

KMDW grid cell indices (`HRRR_GRID_IY=665, HRRR_GRID_IX=1169`) are cached as constants from [herbie.ipynb](herbie.ipynb) §3 — HRRR's Lambert Conformal grid doesn't change between runs, so we don't recompute the nearest-cell lookup.

In [5]:
try:
    from herbie import Herbie
    _HERBIE_OK = True
except ImportError:
    _HERBIE_OK = False
    print("Herbie not installed. HRRR fetcher will raise on call.")
    print("Install: conda install -c conda-forge herbie-data")

# KMDW grid cell in the HRRR Lambert Conformal grid (cached from herbie.ipynb §3).
# These indices are constant -- HRRR's grid doesn't change between runs.
KMDW_LAT = 41.7868
KMDW_LON = -87.7522
HRRR_GRID_IY = 665   # y-index of the cell nearest KMDW
HRRR_GRID_IX = 1169  # x-index


def fxx_for_peak_window(init_dt, station_tz="America/Chicago",
                       peak_start_hour=12, peak_end_hour=16, target_date=None):
    """Compute the fxx (lead-hour) list to cover the local peak heating window.

    Args:
    - init_dt (str | pd.Timestamp): HRRR init time. Assumed UTC if tz-naive.
    - station_tz (str): IANA tz of the local clock. Default America/Chicago.
    - peak_start_hour, peak_end_hour (int): peak window endpoints in local clock
      hours. Defaults match model_v3's PEAK_{START,END}_HOUR_LOCAL = 12, 16.
    - target_date (datetime.date | pd.Timestamp | None): the LOCAL date whose
      peak window to target. If None, defaults to the init's own local date.
      Pass an explicit target_date when the init's local date differs from
      the day we want to forecast (e.g., midnight-anchor case where the 00Z
      init lives in yesterday's local date but we want today's peak window).

    Returns:
    - list[int]: lead hours, e.g. [5, 6, 7, 8, 9] for a 12Z summer init.
    """
    init_dt = pd.Timestamp(init_dt)
    if init_dt.tz is None:
        init_dt = init_dt.tz_localize("UTC")

    if target_date is None:
        target_date_obj = init_dt.tz_convert(station_tz).date()
    else:
        target_date_obj = pd.Timestamp(target_date).date()

    # DST-safe: build local naive timestamps for the peak hours on target_date
    # and then localize. Avoids the "midnight-local + N hours of UTC" pitfall
    # on spring-forward / fall-back days where the wall-clock hour drifts by 1.
    date_str = target_date_obj.strftime("%Y-%m-%d")
    peak_start_local = pd.Timestamp(f"{date_str} {peak_start_hour:02d}:00:00").tz_localize(station_tz)
    peak_end_local   = pd.Timestamp(f"{date_str} {peak_end_hour:02d}:00:00").tz_localize(station_tz)

    fxx_start = int((peak_start_local.tz_convert("UTC") - init_dt).total_seconds() / 3600)
    fxx_end   = int((peak_end_local.tz_convert("UTC")   - init_dt).total_seconds() / 3600)
    return list(range(fxx_start, fxx_end + 1))


def fetch_live_hrrr(init_dt, fxx_list=None, variables=("TMP:2 m",)):
    """Fetch HRRR forecasts for KMDW for one specific run + lead-hour list.

    Args:
    - init_dt (str | pd.Timestamp): HRRR init time, e.g., "2026-05-20 12:00".
      Must be a real published run (use 90+ min buffer after init).
    - fxx_list (iterable[int] | None): forecast hours to pull. If None, auto-
      derived via `fxx_for_peak_window(init_dt)` for the init's local date.
    - variables (iterable[str]): GRIB record search strings. Default just t2m.

    Returns:
    - pd.DataFrame with one row per (init, fxx, variable) at the KMDW grid cell.
    """
    if not _HERBIE_OK:
        raise RuntimeError("Herbie not installed; install with: conda install -c conda-forge herbie-data")

    init_dt = pd.Timestamp(init_dt)
    if fxx_list is None:
        fxx_list = fxx_for_peak_window(init_dt)

    rows = []
    for fxx in fxx_list:
        for var in variables:
            H = Herbie(init_dt.strftime("%Y-%m-%d %H:%M"),
                       model="hrrr", product="sfc", fxx=fxx, verbose=False)
            try:
                ds = H.xarray(var)
            except Exception as e:
                print(f"  HRRR fxx={fxx} {var}: {type(e).__name__}: {e}")
                continue
            var_name = list(ds.data_vars.keys())[0]
            value = float(ds[var_name].isel(y=HRRR_GRID_IY, x=HRRR_GRID_IX).values)
            valid_utc = init_dt + pd.Timedelta(hours=fxx)
            rows.append({
                "init_dt":   init_dt,
                "fxx":       int(fxx),
                "valid_utc": valid_utc,
                "var_name":  var_name,
                "value":     value,
            })
    return pd.DataFrame(rows)


def latest_safe_hrrr_init(now_utc=None, publish_lag_min=90, synoptic_only=True):
    """Pick the latest HRRR init that is guaranteed published as of `now_utc`.

    Args:
    - now_utc (pd.Timestamp | None): reference time. Default = pd.Timestamp.now("UTC").
    - publish_lag_min (int): publish-latency buffer in minutes. Default 90.
    - synoptic_only (bool): if True, restrict to 00/06/12/18Z (48-h horizon).

    Returns:
    - pd.Timestamp (tz-aware UTC): the latest safely-published init.
    """
    if now_utc is None:
        now_utc = pd.Timestamp.now(tz="UTC")
    else:
        now_utc = pd.Timestamp(now_utc)
        if now_utc.tz is None:
            now_utc = now_utc.tz_localize("UTC")

    threshold = now_utc - pd.Timedelta(minutes=publish_lag_min)
    candidate = threshold.floor("h")

    if synoptic_only:
        while candidate.hour not in (0, 6, 12, 18):
            candidate -= pd.Timedelta(hours=1)

    return candidate


def fetch_hrrr_for_anchor(anchor_dt_local, station_tz="America/Chicago",
                         publish_lag_min=90, variables=("TMP:2 m",),
                         now_utc=None):
    """High-level fetch: given a local-time anchor, return HRRR peak-window forecasts.

    Picks the latest HRRR init that is safely published *as of whichever is earlier*:
    the anchor time, or `now_utc`. This handles three cases correctly:
      - anchor in the past (backtest):  uses anchor as as-of -- the run that
                                        would have been available to a model
                                        running at the anchor moment.
      - anchor == now (live trading):   uses anchor (= now) as as-of -- the
                                        latest run available right now.
      - anchor in the future (pre-anchor smoke test): uses `now_utc` as as-of --
                                        the latest run currently published, since
                                        we can only fetch what exists right now.

    The fxx-list derivation always targets the ANCHOR's local date's peak window
    (noon-4 PM local), not the init's local date. This fixes the midnight-anchor
    edge case where the safe 00Z init lives in yesterday's local date.

    Args:
    - anchor_dt_local (str | pd.Timestamp): prediction-time anchor in local clock.
      If tz-naive, assumed in `station_tz`.
    - station_tz (str): IANA tz. Default America/Chicago.
    - publish_lag_min (int): publish-lag buffer. Default 90.
    - variables (iterable[str]): GRIB record search strings.
    - now_utc (pd.Timestamp | None): reference time for "what is currently
      published". Default = pd.Timestamp.now("UTC").

    Returns:
    - (init_dt, fxx_list, df) tuple.
    """
    anchor = pd.Timestamp(anchor_dt_local)
    if anchor.tz is None:
        anchor = anchor.tz_localize(station_tz)
    anchor_utc = anchor.tz_convert("UTC")

    if now_utc is None:
        now_utc = pd.Timestamp.now(tz="UTC")
    else:
        now_utc = pd.Timestamp(now_utc)
        if now_utc.tz is None:
            now_utc = now_utc.tz_localize("UTC")

    # Clamp as-of to min(anchor, now): can only fetch what currently exists,
    # but also not exceed the run that would be available at the anchor time.
    as_of_utc = min(anchor_utc, now_utc)

    init_dt = latest_safe_hrrr_init(as_of_utc, publish_lag_min=publish_lag_min)

    # Pin the peak window to the ANCHOR's local date (not the init's local date,
    # which can differ for the midnight-anchor case where init=00Z lives in
    # yesterday's local calendar).
    target_date = anchor.date()
    fxx_list = fxx_for_peak_window(init_dt, station_tz=station_tz, target_date=target_date)

    df = fetch_live_hrrr(init_dt, fxx_list=fxx_list, variables=variables)
    return init_dt, fxx_list, df


## §2.5 — Live CLI (IEM)

CLI is the settlement source — not a prediction-time input. We fetch it for
two reasons in live trading:

1. **Persistence features:** `cli_high_yesterday`, `cli_high_lag_mean_3d`, etc.
   need yesterday's settled CLI to be populated for today's feature row.
2. **Settlement reconciliation:** after the day's CLI publishes (next morning),
   confirm Kalshi's settlement matches and log realized P&L.

IEM serves CLI as JSON one year at a time. For live use, just pull the current
year and filter to the most recent days.

In [6]:
def fetch_live_cli(station=STATION, days_back=30):
    """Fetch recent CLI daily-max reports from IEM.

    Args:
    - station (str): full ICAO (e.g., 'KMDW').
    - days_back (int): how many recent days to return. Default 30.

    Returns:
    - pd.DataFrame matching data/cli_KMDW.parquet schema, last `days_back` rows.
    """
    now = datetime.now(timezone.utc)
    start = now - timedelta(days=days_back)
    df = _hist_fetch_cli(
        start.strftime("%m/%d/%Y"),
        (now + timedelta(days=1)).strftime("%m/%d/%Y"),
        station,
    )
    return _coerce_to_reference(df, DATA_DIR / f"cli_{station}.parquet")


## §2.6 — Live Kalshi market state (api.elections.kalshi.com)

Fetches the 6-contract KXHIGHCHI event for a given settlement date. Returns
strikes, bid/ask prices, volume, open interest — everything needed to compute
edge against our model's PMF.

**No authentication required for market data**. Order placement (Phase 2)
would require a paid API key — out of scope here.

The event ticker follows the pattern `KXHIGHCHI-YYMMMDD`. E.g., settlement
date 2026-05-20 → event_ticker `KXHIGHCHI-26MAY20` → six market tickers
under it (1× T-greater + 1× T-less + 4× B-pairs).

In [7]:
def fetch_kalshi_market(date=None):
    """Fetch the 6-contract Kalshi KXHIGHCHI event for one settlement date.

    Args:
    - date (str | pd.Timestamp | None): settlement date. Defaults to today
      (local America/Chicago).

    Returns:
    - pd.DataFrame with one row per contract (6 rows for a standard day):
        ticker, strike_type, strike, subtitle,
        yes_bid_cents, yes_ask_cents, no_bid_cents, no_ask_cents,
        last_price_cents, volume_24h, open_interest,
        open_time_utc, close_time_utc, status.
      Sorted so the 4 middle B-pair contracts are in ascending-strike order
      and the two T edges bracket them.
    """
    if date is None:
        date = pd.Timestamp.now(tz="America/Chicago").normalize()
    else:
        date = pd.Timestamp(date)

    event_ticker = f"KXHIGHCHI-{date.strftime('%y%b%d').upper()}"
    url = f"{KALSHI_API}/markets?event_ticker={event_ticker}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    markets = payload.get("markets", [])

    if not markets:
        return pd.DataFrame()

    rows = []
    for m in markets:
        rows.append({
            "ticker":           m.get("ticker"),
            "strike_type":      m.get("strike_type"),       # "greater" / "less" / None for B-pairs
            "strike":           m.get("floor_strike"),       # the T-strike value (None for B)
            "subtitle":         m.get("subtitle"),           # human-readable, e.g. "82° to 83°"
            "yes_bid_cents":    int(round(float(m.get("yes_bid_dollars", 0)) * 100)),
            "yes_ask_cents":    int(round(float(m.get("yes_ask_dollars", 1)) * 100)),
            "no_bid_cents":     int(round(float(m.get("no_bid_dollars", 0)) * 100)),
            "no_ask_cents":     int(round(float(m.get("no_ask_dollars", 1)) * 100)),
            "last_price_cents": int(round(float(m.get("last_price_dollars", 0)) * 100)),
            "volume_24h":       m.get("volume_24h_fp"),
            "open_interest":    m.get("open_interest_fp"),
            "open_time_utc":    pd.to_datetime(m.get("open_time"),  utc=True, errors="coerce"),
            "close_time_utc":   pd.to_datetime(m.get("close_time"), utc=True, errors="coerce"),
            "status":           m.get("status"),
        })
    df = pd.DataFrame(rows)
    # Sort: T-less (low edge) first, then B-pairs by strike, then T-greater (high edge)
    def _sort_key(row):
        if row["strike_type"] == "less":
            return -1
        if row["strike_type"] == "greater":
            return 999
        # B-pair: parse "X° to X+1°" into low integer
        try:
            return int(row["subtitle"].split("°")[0])
        except Exception:
            return 500
    df["_sort"] = df.apply(_sort_key, axis=1)
    return df.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)


## §2.7 — Schema-parity assertions (train-serve boundary)

Each `fetch_live_*` function returns a DataFrame that the §3 feature
engineering cells (from `model_v*.ipynb`) consume. If the live data's
columns or dtypes diverge from what the model was trained on, feature
engineering produces subtly wrong inputs and predictions silently drift.

`assert_schema_parity(live_df, reference_parquet)` reads the historical
parquet's columns + dtypes once at startup and asserts the live DataFrame
matches exactly. Cheap (a few ms per check) and catches the failure mode
immediately rather than after days of bad trades.

Applies to the four data sources with a historical parquet:
**METAR, TAF, ASOS, CLI**. HRRR and Kalshi don't have a historical cache
yet, so they're skipped (HRRR uses a fixed in-code schema; Kalshi is
fetched live-only).

In [8]:
def assert_schema_parity(live_df, reference_parquet, name="", strict_dtype=True):
    """Assert live_df has the same columns and dtypes as the reference parquet.

    Args:
    - live_df (pd.DataFrame): the freshly-fetched live DataFrame.
    - reference_parquet (str | Path): path to the historical parquet the model
      was trained on (e.g., "data/metar_KMDW.parquet").
    - name (str): short label used in error messages (e.g., "metar").
    - strict_dtype (bool): if True, dtype strings must match exactly. If False,
      compatible numeric types (Int64 vs int64, etc.) are tolerated. Default True.

    Raises:
    - AssertionError with a structured diagnostic if columns or dtypes differ.

    Returns:
    - None on success.
    """
    ref = pd.read_parquet(reference_parquet)

    missing = set(ref.columns) - set(live_df.columns)
    extra   = set(live_df.columns) - set(ref.columns)
    if missing or extra:
        raise AssertionError(
            f"[{name}] column mismatch vs {Path(reference_parquet).name}:\n"
            f"  missing in live: {sorted(missing)}\n"
            f"  extra in live:   {sorted(extra)}"
        )

    if strict_dtype:
        mismatches = []
        for col in ref.columns:
            ref_dt  = str(ref[col].dtype)
            live_dt = str(live_df[col].dtype)
            if ref_dt != live_dt:
                mismatches.append((col, live_dt, ref_dt))
        if mismatches:
            lines = "\n".join(f"  {col:30s}  live={lv:15s}  ref={rv}" for col, lv, rv in mismatches)
            raise AssertionError(
                f"[{name}] dtype mismatch vs {Path(reference_parquet).name}:\n{lines}"
            )

    return None


## §2.8 — Concurrent refresh wrapper

`refresh_for_live(anchor_dt_local, station, sources)` runs the six fetchers
**in parallel** via a `ThreadPoolExecutor`. The fetchers are I/O-bound (each
spends most of its time waiting for an HTTPS response), so threads — not
processes — are the right primitive. Expected wall-clock speedup vs serial:
roughly the time of the slowest single fetcher (HRRR), since the others
finish in parallel under its umbrella.

Returns a dict `{source: DataFrame}` plus a `_wall_time` key reporting how
long the whole concurrent batch took. HRRR's tuple `(init_dt, fxx_list, df)`
is split: `results["hrrr"]` is the DataFrame, `results["hrrr_meta"]` is a
dict with the init / fxx info.

This is the single entrypoint a `live_predict.ipynb` notebook would call
at each anchor — one function, one wait, all six sources current.

In [9]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def refresh_for_live(anchor_dt_local=None, station=STATION, sources=None,
                     hrrr_now_utc=None, verbose=True):
    """Run all live fetchers concurrently. Returns a dict of source -> DataFrame.

    METAR special case: fetch_live_metar() returns a dict
    {"tgroup": df, "obs5min": df}; the same dict is stored under results["metar"].
    Downstream consumers should pick the subset they want explicitly.

    Args:
    - anchor_dt_local (str | pd.Timestamp | None): local-time prediction anchor.
      Required for HRRR; if None, HRRR is skipped.
    - station (str): ICAO code. Default STATION (KMDW).
    - sources (iterable[str] | None): subset of {metar, taf, asos, cli, hrrr,
      kalshi}. Default = all six (minus hrrr if anchor_dt_local is None).
    - hrrr_now_utc (pd.Timestamp | None): passed through to fetch_hrrr_for_anchor
      to override the "now" reference. Useful for backtest scenarios.
    - verbose (bool): print per-source success / failure.

    Returns:
    - dict with keys for each requested source (value = DataFrame or None on
      failure, or the {"tgroup", "obs5min"} dict for "metar") plus:
        - "hrrr_meta" -> {"init_dt": ..., "fxx_list": ...} when HRRR ran
        - "_wall_time" -> total wall-clock seconds for the concurrent batch
    """
    DEFAULT_SOURCES = ["metar", "taf", "asos", "cli", "hrrr", "kalshi"]
    if sources is None:
        sources = list(DEFAULT_SOURCES)
    sources = list(sources)
    if anchor_dt_local is None and "hrrr" in sources:
        sources.remove("hrrr")
        if verbose:
            print("  [refresh_for_live] no anchor passed; skipping HRRR.")

    def _hrrr_call():
        return fetch_hrrr_for_anchor(anchor_dt_local, now_utc=hrrr_now_utc)

    task_map = {
        "metar":  lambda: fetch_live_metar(station, 40),
        "taf":    lambda: fetch_live_taf(station, 36),
        "asos":   lambda: fetch_live_asos_1min(station, 48),
        "cli":    lambda: fetch_live_cli(station, 10),
        "hrrr":   _hrrr_call,
        "kalshi": lambda: fetch_kalshi_market(None),
    }

    results = {}
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=len(sources)) as exe:
        fut_to_name = {exe.submit(task_map[name]): name for name in sources}
        for fut in as_completed(fut_to_name):
            name = fut_to_name[fut]
            try:
                result = fut.result()
                if name == "hrrr":
                    init_dt, fxx_list, df = result
                    results["hrrr"] = df
                    results["hrrr_meta"] = {"init_dt": init_dt, "fxx_list": fxx_list}
                else:
                    results[name] = result
                if verbose:
                    r = results.get(name)
                    if isinstance(r, dict):
                        parts = " ".join(
                            f"{k}={len(v)}" for k, v in r.items()
                            if isinstance(v, pd.DataFrame)
                        )
                        print(f"  [{name:6s}] ok  {parts}")
                    else:
                        rows = len(r) if r is not None else "n/a"
                        print(f"  [{name:6s}] ok  rows={rows}")
            except Exception as e:
                results[name] = None
                if verbose:
                    print(f"  [{name:6s}] FAILED: {type(e).__name__}: {e}")

    results["_wall_time"] = time.perf_counter() - t0
    if verbose:
        print(f"  Total wall time: {results['_wall_time']:.2f} s "
              f"({len(sources)} sources, max {len(sources)} workers)")
    return results


## §3 — Smoke tests

Hit each fetcher once and report shape + freshness. If everything works, the
notebook is ready to slot into a live prediction pipeline (next step:
`live_predict.ipynb` or similar that combines §2 fetchers + model_v3.predict +
edge calculation + decision logic).

In [10]:
# §2.1 -- METAR via api.weather.gov (dict-return: tgroup + obs5min)
print("=" * 60)
print("§2.1 fetch_live_metar")
print("=" * 60)
_t0 = time.perf_counter()
metar_live = fetch_live_metar(STATION, limit=40)
_dt = time.perf_counter() - _t0
tgroup  = metar_live["tgroup"]
obs5min = metar_live["obs5min"]
print(f"  pull duration: {_dt:.2f} s")
print(f"  tgroup  rows: {len(tgroup):>3}  (hourly METAR/SPECI with rawMessage, 0.1C precision)")
print(f"  obs5min rows: {len(obs5min):>3}  (all observations incl. 5-min)")
if len(obs5min):
    latest = obs5min.iloc[-1]
    age_min = (NOW_UTC - latest["timestamp"]).total_seconds() / 60
    print(f"  freshest obs:    {latest['timestamp']}  ({age_min:5.1f} min ago, temp_c={latest.get('temp_c')})")
if len(tgroup):
    latest_t = tgroup.iloc[-1]
    age_min_t = (NOW_UTC - latest_t["timestamp"]).total_seconds() / 60
    print(f"  freshest T-grp:  {latest_t['timestamp']}  ({age_min_t:5.1f} min ago, temp_c={latest_t.get('temp_c')})")


§2.1 fetch_live_metar
  pull duration: 0.65 s
  tgroup  rows:   3  (hourly METAR/SPECI with rawMessage, 0.1C precision)
  obs5min rows:  40  (all observations incl. 5-min)
  freshest obs:    2026-05-22 06:20:00+00:00  ( 20.6 min ago, temp_c=13.0)
  freshest T-grp:  2026-05-22 05:53:00+00:00  ( 47.6 min ago, temp_c=12.8)


In [11]:
# §2.2 -- TAF via IEM
print("=" * 60)
print("§2.2 fetch_live_taf")
print("=" * 60)
_t0 = time.perf_counter()
taf_live = fetch_live_taf(STATION, hours_back=36)
_dt = time.perf_counter() - _t0
print(f"  pull duration: {_dt:.2f} s")
print(f"  rows: {len(taf_live)}")
if len(taf_live):
    latest_valid = taf_live["valid"].max()
    age_h = (NOW_UTC - latest_valid).total_seconds() / 3600
    print(f"  most recent TAF issuance: {latest_valid}  ({age_h:.1f} h ago)")
    n_issuances = taf_live["valid"].nunique()
    print(f"  distinct issuances in window: {n_issuances}")


§2.2 fetch_live_taf
  pull duration: 0.41 s
  rows: 46
  most recent TAF issuance: 2026-05-22 05:32:00+00:00  (1.1 h ago)
  distinct issuances in window: 13


In [12]:
# §2.3 -- ASOS 1-min via IEM (with lag caveat)
print("=" * 60)
print("§2.3 fetch_live_asos_1min")
print("=" * 60)
_t0 = time.perf_counter()
asos_live = fetch_live_asos_1min(STATION, hours_back=48)
_dt = time.perf_counter() - _t0
print(f"  pull duration: {_dt:.2f} s")
print(f"  rows: {len(asos_live)}")
if len(asos_live):
    latest = asos_live.iloc[-1]
    age_min = (NOW_UTC - latest["timestamp"]).total_seconds() / 60
    print(f"  latest: {latest['timestamp']}  ({age_min:.1f} min ago)")
    print(f"  latest temp_f: {latest.get('temp_f')}")
    if age_min > 60:
        print(f"  WARNING: lag exceeds 60 min -- last hour of obs unavailable")


§2.3 fetch_live_asos_1min
  pull duration: 1.03 s
  rows: 1330
  latest: 2026-05-20 22:09:00+00:00  (1951.6 min ago)
  latest temp_f: 57


In [13]:
# §2.4 -- HRRR via Herbie/AWS (anchor-aware, future-safe, midnight-safe)
print("=" * 60)
print("§2.4 fetch_live_hrrr (anchor-aware, future-safe)")
print("=" * 60)

now_utc = pd.Timestamp.now(tz="UTC")
print(f"  now_utc: {now_utc}")

# 1. Demonstrate latest_safe_hrrr_init for "now"
safe_init_now = latest_safe_hrrr_init(now_utc)
age_h = (now_utc - safe_init_now).total_seconds() / 3600
print(f"  latest_safe_hrrr_init (as-of now): {safe_init_now}  ({age_h:.1f} h old, synoptic)")
print()

# 2. Anchor-aware fetch for 1 PM CDT today (often in the future when running early)
anchor_local = pd.Timestamp.now(tz="America/Chicago").normalize() + pd.Timedelta(hours=13)
hours_to_anchor = (anchor_local.tz_convert("UTC") - now_utc).total_seconds() / 3600
print(f"  anchor (1 PM local): {anchor_local}  ({hours_to_anchor:+.1f} h relative to now)")

try:
    _t0 = time.perf_counter()
    init_dt, fxx_list, hrrr_live = fetch_hrrr_for_anchor(anchor_local)
    _dt = time.perf_counter() - _t0
    print(f"  -> selected init: {init_dt}  (fxx={fxx_list})")
    print(f"  pull duration: {_dt:.2f} s  ({_dt/max(1,len(fxx_list)):.2f} s per fxx)")
    print(f"  rows: {len(hrrr_live)}")
    if len(hrrr_live):
        hrrr_live["t_f"] = (hrrr_live["value"] - 273.15) * 9 / 5 + 32
        print(hrrr_live[["fxx", "valid_utc", "var_name", "value", "t_f"]].to_string(index=False))
        print(f"  peak forecast (max across fxx, in °F): {hrrr_live['t_f'].max():.1f}°F")
except Exception as e:
    print(f"  FAILED: {type(e).__name__}: {e}")

print()

# 3. Sanity-check the midnight-anchor edge case (target_date pinning)
print("  -- midnight-anchor sanity check (no fetch, just lead-list) --")
midnight_local = pd.Timestamp.now(tz="America/Chicago").normalize()   # 00:00 local today
midnight_utc = midnight_local.tz_convert("UTC")
# Pretend "now" is right at midnight so we exercise the edge case cleanly
mid_init = latest_safe_hrrr_init(midnight_utc)
mid_fxx_no_override = fxx_for_peak_window(mid_init)
mid_fxx_with_override = fxx_for_peak_window(mid_init, target_date=midnight_local.date())
print(f"  midnight_local: {midnight_local}")
print(f"  safe_init at midnight: {mid_init}  (local date = {mid_init.tz_convert('America/Chicago').date()})")
print(f"  fxx without target_date override: {mid_fxx_no_override}  <- would be wrong if init's local date != anchor's date")
print(f"  fxx with    target_date override: {mid_fxx_with_override}  <- correct (anchor's day)")


§2.4 fetch_live_hrrr (anchor-aware, future-safe)
  now_utc: 2026-05-22 06:40:45.224247+00:00
  latest_safe_hrrr_init (as-of now): 2026-05-22 00:00:00+00:00  (6.7 h old, synoptic)

  anchor (1 PM local): 2026-05-22 13:00:00-05:00  (+11.3 h relative to now)
  -> selected init: 2026-05-22 00:00:00+00:00  (fxx=[17, 18, 19, 20, 21])
  pull duration: 6.66 s  (1.33 s per fxx)
  rows: 5
 fxx                 valid_utc var_name      value       t_f
  17 2026-05-22 17:00:00+00:00      t2m 291.031860 64.187349
  18 2026-05-22 18:00:00+00:00      t2m 292.474823 66.784681
  19 2026-05-22 19:00:00+00:00      t2m 293.627533 68.859559
  20 2026-05-22 20:00:00+00:00      t2m 294.269775 70.015596
  21 2026-05-22 21:00:00+00:00      t2m 294.468994 70.374189
  peak forecast (max across fxx, in °F): 70.4°F

  -- midnight-anchor sanity check (no fetch, just lead-list) --
  midnight_local: 2026-05-22 00:00:00-05:00
  safe_init at midnight: 2026-05-22 00:00:00+00:00  (local date = 2026-05-21)
  fxx without tar

In [14]:
# §2.5 -- CLI via IEM
print("=" * 60)
print("§2.5 fetch_live_cli")
print("=" * 60)
_t0 = time.perf_counter()
cli_live = fetch_live_cli(STATION, days_back=10)
_dt = time.perf_counter() - _t0
print(f"  pull duration: {_dt:.2f} s")
print(f"  rows: {len(cli_live)}")
if len(cli_live):
    print(cli_live.tail(5).to_string(index=False))


§2.5 fetch_live_cli
  pull duration: 0.54 s
  rows: 10
station       date  max_temp_f
   KMDW 2026-05-17          87
   KMDW 2026-05-18          80
   KMDW 2026-05-19          79
   KMDW 2026-05-20          61
   KMDW 2026-05-21          61


In [15]:
# §2.6 -- Kalshi KXHIGHCHI for today
print("=" * 60)
print("§2.6 fetch_kalshi_market (today)")
print("=" * 60)
today_chi = pd.Timestamp.now(tz="America/Chicago").normalize()
_t0 = time.perf_counter()
kalshi = fetch_kalshi_market(today_chi)
_dt = time.perf_counter() - _t0
print(f"  pull duration: {_dt:.2f} s")
print(f"  rows: {len(kalshi)}")
if len(kalshi):
    cols = ["ticker", "subtitle", "yes_bid_cents", "yes_ask_cents",
            "no_bid_cents", "no_ask_cents", "last_price_cents",
            "volume_24h", "status"]
    print(kalshi[cols].to_string(index=False))
    print()
    close = kalshi["close_time_utc"].dropna().min()
    if pd.notna(close):
        hrs_to_close = (close - NOW_UTC).total_seconds() / 3600
        print(f"  market closes in: {hrs_to_close:.1f} h ({close})")
else:
    print(f"  No KXHIGHCHI event for {today_chi.date()} (market may not be listed yet)")


§2.6 fetch_kalshi_market (today)
  pull duration: 0.15 s
  rows: 6
                 ticker     subtitle  yes_bid_cents  yes_ask_cents  no_bid_cents  no_ask_cents  last_price_cents volume_24h status
  KXHIGHCHI-26MAY22-T64 63° or below              0              2            98           100                 1    4263.38 active
KXHIGHCHI-26MAY22-B64.5   64° to 65°              8             10            90            92                 7    1771.67 active
KXHIGHCHI-26MAY22-B66.5   66° to 67°             28             29            71            72                30    3611.27 active
KXHIGHCHI-26MAY22-B68.5   68° to 69°             36             37            63            64                36    2334.20 active
KXHIGHCHI-26MAY22-B70.5   70° to 71°             19             20            80            81                18    2958.79 active
  KXHIGHCHI-26MAY22-T71 72° or above              3              4            96            97                 4    1922.28 active

  market closes

In [16]:
# §3.7 -- schema-parity assertions vs the historical parquets
print("=" * 60)
print("§3.7 schema-parity assertions")
print("=" * 60)

PARITY_CHECKS = [
    ("metar.tgroup",  metar_live["tgroup"],  f"data/metar_{STATION}.parquet"),
    ("metar.obs5min", metar_live["obs5min"], f"data/metar_{STATION}.parquet"),
    ("taf",   taf_live,   f"data/taf_{STATION}.parquet"),
    ("asos",  asos_live,  f"data/asos_{STATION}.parquet"),
    ("cli",   cli_live,   f"data/cli_{STATION}.parquet"),
]

for name, live_df, ref_path in PARITY_CHECKS:
    try:
        assert_schema_parity(live_df, ref_path, name=name)
        print(f"  [{name:14s}] schema parity OK  ({len(live_df.columns)} cols)")
    except AssertionError as e:
        print(f"  [{name:14s}] FAILED")
        for line in str(e).split("\n"):
            print(f"    {line}")


§3.7 schema-parity assertions
  [metar.tgroup  ] schema parity OK  (27 cols)
  [metar.obs5min ] schema parity OK  (27 cols)
  [taf           ] schema parity OK  (19 cols)
  [asos          ] schema parity OK  (10 cols)
  [cli           ] schema parity OK  (3 cols)


In [18]:
# §3.8 -- concurrent refresh wrapper
print("=" * 60)
print("§3.8 refresh_for_live (concurrent)")
print("=" * 60)
anchor_local = pd.Timestamp.now(tz="America/Chicago").normalize() + pd.Timedelta(hours=13)
print(f"  anchor: {anchor_local}")
print()

results = refresh_for_live(anchor_dt_local=anchor_local, verbose=True)

print()
# Sanity-check rows per source. METAR returns a {tgroup, obs5min} dict, so it
# gets a nested summary line.
print("  Returned DataFrames:")
for k in ("metar", "taf", "asos", "cli", "hrrr", "kalshi"):
    df = results.get(k)
    if df is None:
        print(f"    {k:6s}: None")
    elif isinstance(df, dict):
        for sk, sv in df.items():
            if isinstance(sv, pd.DataFrame):
                print(f"    {k:6s}.{sk:7s}: rows={len(sv):>5}  cols={len(sv.columns)}")
    else:
        print(f"    {k:6s}: rows={len(df):>5}  cols={len(df.columns)}")

if "hrrr_meta" in results:
    meta = results["hrrr_meta"]
    print(f"  hrrr_meta: init={meta['init_dt']}  fxx={meta['fxx_list']}")


§3.8 refresh_for_live (concurrent)
  anchor: 2026-05-22 13:00:00-05:00

  [kalshi] ok  rows=6
  [metar ] ok  tgroup=3 obs5min=40
  [taf   ] ok  rows=46
  [cli   ] ok  rows=10
  [asos  ] ok  rows=1330
  [hrrr  ] ok  rows=5
  Total wall time: 3.77 s (6 sources, max 6 workers)

  Returned DataFrames:
    metar .tgroup : rows=    3  cols=27
    metar .obs5min: rows=   40  cols=27
    taf   : rows=   46  cols=19
    asos  : rows= 1330  cols=10
    cli   : rows=   10  cols=3
    hrrr  : rows=    5  cols=5
    kalshi: rows=    6  cols=14
  hrrr_meta: init=2026-05-22 00:00:00+00:00  fxx=[17, 18, 19, 20, 21]


## §4 — Next steps once §2.1–§2.6 all work cleanly

1. **Schema validation:** for each live fetcher, write a quick assertion that
   the returned DataFrame has the same columns and dtypes as the corresponding
   historical parquet. Catch any divergence early.
2. **Combined refresh function:** `refresh_for_live(date)` that calls all six
   fetchers in parallel (via `concurrent.futures`) and returns a dict of
   DataFrames. Single entrypoint for the live prediction notebook.
3. **Build `live_predict.ipynb`:**
   - §1 setup + import model_v3 artifacts
   - §2 call refresh_for_live() and rebuild today's feature row
   - §3 model_v3.predict(today) → integer PMF
   - §4 aggregate PMF into Kalshi 6 buckets using strikes from §2.6
   - §5 compute per-bucket edge = P_model - P_market_mid
   - §6 decide trades (edge > fees threshold)
   - §7 log decision + market state to parquet
4. **Paper trading loop:** schedule the above to run at each anchor time
   (cron job or scheduled Jupyter execution); log everything to `data/live_log.parquet`.
   Run for 30+ days before going live with real orders.